# V09 - NodeApply e escrita segura no modelo

**Objetivo:** criar, atualizar e recuperar um `NodeApply` tipado em um contrato DMS temporario, mostrando **alteracao** versus **duplicacao**.

**Escrita segura:** a chave `space + external_id` garante que reaplicar atualiza o mesmo node; mudar uma propriedade altera, nao duplica.

In [ ]:
# Carrega a configuracao local e cria o cliente CDF. / Load local configuration and create the CDF client.
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from cognite.client import ClientConfig, CogniteClient
from cognite.client.credentials import OAuthClientCredentials, Token

env_file = Path('.env')
pkg_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env.example').exists()),
    Path.cwd(),
)

if env_file.exists():
    load_dotenv(env_file)
else:
    for name in ('.env', '.env.example'):
        candidate = pkg_root / name
        if candidate.exists():
            load_dotenv(candidate)
            break

if not os.environ.get('CDF_CLUSTER'):
    os.environ['CDF_CLUSTER'] = input('CDF_CLUSTER: ').strip()
if not os.environ.get('CDF_PROJECT'):
    os.environ['CDF_PROJECT'] = input('CDF_PROJECT: ').strip()

base_url = os.environ.get('CDF_URL', '').rstrip('/') or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
oauth_ready = env_file.exists() and all(
    os.environ.get(key) for key in ('IDP_TOKEN_URL', 'IDP_CLIENT_ID', 'IDP_CLIENT_SECRET')
)
if oauth_ready:
    scopes = os.environ.get('IDP_SCOPES', '').replace(',', ' ').split() or [f'{base_url}/.default']
    audience = os.environ.get('IDP_AUDIENCE', '')
    credentials = OAuthClientCredentials(
        token_url=os.environ['IDP_TOKEN_URL'],
        client_id=os.environ['IDP_CLIENT_ID'],
        client_secret=os.environ['IDP_CLIENT_SECRET'],
        scopes=scopes,
        **({'audience': audience} if audience else {}),
    )
else:
    # Sem .env nesta pasta: usa Bearer Token.
    bearer_token = (os.environ.get('CDF_BEARER_TOKEN') or getpass('CDF_BEARER_TOKEN: ')).strip()
    if bearer_token.lower().startswith('bearer '):
        bearer_token = bearer_token[7:].strip()
    credentials = Token(bearer_token)

client = CogniteClient(ClientConfig(
    client_name='radix-cdf-onboarding-v09',
    base_url=base_url,
    project=os.environ['CDF_PROJECT'],
    credentials=credentials,
))

## 1. Criar contrato tipado (container + view)

In [ ]:
from uuid import uuid4
from cognite.client.data_classes.data_modeling import (
    ContainerApply, ContainerId, MappedPropertyApply, NodeApply, NodeId,
    NodeOrEdgeData, SpaceApply, Text, ViewApply, ViewId,
)

training_space = f'sp_ur_training_v09_{uuid4().hex[:8]}'
container_id = ContainerId(training_space, 'Equipment')
view_id = ViewId(training_space, 'Equipment', 'v1')
node_id = NodeId(training_space, 'equipment-001')

client.data_modeling.spaces.apply(SpaceApply(space=training_space, name='UR training - V09'))
client.data_modeling.containers.apply(ContainerApply._load({
    'space': training_space,
    'externalId': container_id.external_id,
    'properties': {'name': {'type': Text().dump(), 'nullable': False}},
    'usedFor': 'node',
}))
client.data_modeling.views.apply(ViewApply(
    space=training_space, external_id=view_id.external_id, version=view_id.version,
    properties={'name': MappedPropertyApply(container=container_id, container_property_identifier='name')},
))
print('contrato criado em', training_space)

## 2. Primeira escrita
Aplica o node com `name='initial'`.

In [ ]:
payload = NodeApply(
    space=training_space,
    external_id=node_id.external_id,
    sources=[NodeOrEdgeData(source=view_id, properties={'name': 'initial'})],
)
first_apply = client.data_modeling.instances.apply(nodes=payload)
first_apply

## 3. Alterar (nao duplicar)
Recupera, muda a propriedade e reaplica; a contagem do node nao aumenta.

In [ ]:
node = client.data_modeling.instances.retrieve_nodes(nodes=node_id, sources=view_id)
assert node is not None
payload = node.as_apply()
assert payload.sources is not None
payload.sources[0].properties['name'] = 'updated'
second_apply = client.data_modeling.instances.apply(nodes=payload)
updated_node = client.data_modeling.instances.retrieve_nodes(nodes=node_id, sources=view_id)
assert updated_node is not None
assert updated_node.properties[view_id]['name'] == 'updated'
count = len(client.data_modeling.instances.list(instance_type='node', space=training_space, limit=10))
print('valor atual:', updated_node.properties[view_id]['name'], '| nodes no space:', count)
updated_node

## Mini-exercicio
- Adicione `status` (Text) ao contrato e atualize apenas esse campo, mantendo `name`.
- Use `instances.aggregate` (ou `len(list(...))`) para confirmar que continua existindo 1 node.

## 4. Limpeza idempotente

In [ ]:
assert training_space.startswith('sp_ur_training_v09_')
client.data_modeling.instances.delete(nodes=node_id)
client.data_modeling.views.delete(view_id)
client.data_modeling.containers.delete(container_id)
client.data_modeling.spaces.delete(training_space)
print('space_still_exists:', client.data_modeling.spaces.retrieve(training_space) is not None)